In [16]:
from textblob import TextBlob
import pandas as pd
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

In [2]:
df = pd.read_csv('bank_reviews.csv')

In [7]:
def classify_sentiment(text):
    analysis = TextBlob(text)
    if analysis.sentiment.polarity > 0:
        return 'positive'
    elif analysis.sentiment.polarity < 0:
        return 'negative'
    else:
        return 'neutral'

In [8]:
# Apply sentiment classification
df['sentiment_label'] = df['review'].apply(classify_sentiment)
df['sentiment_score'] = df['review'].apply(lambda x: TextBlob(x).sentiment.polarity)

In [11]:
sentiment_summary

,bank,rating,mean_sentiment,count
0,BOA,1,-0.120493,398
1,BOA,2,-0.017769,47
2,BOA,3,0.046248,54
3,BOA,4,0.250014,32
4,BOA,5,0.300447,256
5,CBE,1,-0.046804,1038
6,CBE,2,0.073307,277
7,CBE,3,0.177372,400
8,CBE,4,0.312377,601
9,CBE,5,0.364903,3206


In [10]:
sentiment_summary.to_csv('sentiment_summary.csv', index=False)

In [18]:
nlp = spacy.load("en_core_web_sm")

In [19]:
# Preprocess reviews for thematic analysis
def preprocess_text(text):
    doc = nlp(text)
    tokens = [token.lemma_ for token in doc if not token.is_stop and not token.is_punct]
    return ' '.join(tokens)

In [20]:
# Apply preprocessing
df['processed_review'] = df['review'].apply(preprocess_text)

In [21]:
# Extract keywords using TF-IDF
vectorizer = TfidfVectorizer(max_features=1000)
X = vectorizer.fit_transform(df['processed_review'])

# Get feature names
features = vectorizer.get_feature_names_out()
dense = X.todense()
denselist = dense.tolist()
tfidf_df = pd.DataFrame(denselist, columns=features)

In [22]:
# Get top keywords per bank
top_keywords = {}
for bank in df['bank'].unique():
    bank_reviews = tfidf_df[df['bank'] == bank]
    mean_tfidf = bank_reviews.mean().sort_values(ascending=False)
    top_keywords[bank] = mean_tfidf.head(10).index.tolist()

In [23]:
# Define themes based on extracted keywords
themes = {
    "CBE": ["Account Access Issues", "Transaction Performance", "User Interface & Experience"],
    "BOA": ["Customer Support", "Feature Requests", "Transaction Speed"],
    "Dashen": ["App Stability", "Security Concerns", "User Experience"]
}

In [24]:
# Save keywords and themes
keywords_df = pd.DataFrame({
    'bank': list(top_keywords.keys()),
    'keywords': list(top_keywords.values()),
    'themes': [themes[bank] for bank in top_keywords.keys()]
})

keywords_df.to_csv('keywords_themes.csv', index=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Sentiment distribution
plt.figure(figsize=(10, 6))
sns.countplot(data=reviews_df, x='sentiment', order=reviews_df['sentiment'].value_counts().index)
plt.title('Sentiment Distribution of Reviews')
plt.xlabel('Sentiment')
plt.ylabel('Number of Reviews')
plt.savefig('sentiment_distribution.png')
plt.show()

# Average rating by bank
avg_rating = reviews_df.groupby('bank')['rating'].mean().reset_index()
plt.figure(figsize=(10, 6))
sns.barplot(data=avg_rating, x='bank', y='rating', palette='viridis')
plt.title('Average Rating by Bank')
plt.xlabel('Bank')
plt.ylabel('Average Rating')
plt.savefig('average_rating_by_bank.png')
plt.show()

{'CBE': ['Account Access Issues',
  'Transaction Performance',
  'User Interface & Experience'],
 'BOA': ['Customer Support', 'Feature Requests', 'Transaction Speed'],
 'Dashen': ['App Stability', 'Security Concerns', 'User Experience']}